# Cleaning — FPA-FOD 6th Edition

Turns the raw `Fires` table into the two artifacts the analysis runs on: applies one row
exclusion, attaches three derived analysis keys, and writes:

- **`data/fires_clean.parquet`** — fire-level, one row per fire, with the analysis keys attached.
- **`data/region_season_cause.parquet`** — the analysis grain: burned acres per
  **region × season-year × cause**, the input to RQ1's comparison and RQ2's model.

## What is dropped

**Puerto Rico, Hawaii, and the `IA` reporting stream**, established in
[`03_missingness.ipynb`](03_missingness.ipynb). Cause is missing for 98%+ of PR (98.7%) and HI
(98.1%) fires, and 99.8% of the `IA` stream, which is almost entirely Puerto Rico. These are
reporting streams that never carried cause data, not case-by-case undetermined cause. Since the
deliverable is cause composition weighted by burned area, a region with no cause signal contributes
noise. Together they are ~2.1% of rows but only ~0.26% of burned acres.

This is the only exclusion.

## What is *not* dropped: Alaska

Alaska has no cause-quality problem and carries **20.4% of all burned acres** (36.7M of 180M).

The obstacle was the **region key**, not data quality: the EPA Level III shapefile in
`data/us_eco_l3_state_boundaries/` is the *conterminous* US layer, so Alaskan points fall outside
it and were dropped by the spatial join.

This notebook joins a **second layer**, `data/ak_eco_l3/` (EPA Level III Ecoregions of Alaska, 20
ecoregions, 366 polygons). Its attribute schema matches the CONUS layer on the hierarchy columns,
so Alaska enters the same `region` column at the same Level III grain. The join recovers 97.9% of
AK fires and **99.9% of AK acres**.

The two layers are in different projections (CONUS Albers vs NAD83 Alaska Albers), so each join
runs in its own layer's CRS with the fire points reprojected to match. Neither polygon layer is
reprojected — forcing Alaska into CONUS Albers would distort it badly at those latitudes.

## Derived keys

- **`region`** — `US_L3NAME`, the Level III ecoregion name (unique across both layers). `na_l2name`
  is carried alongside so the grain can be coarsened to Level II without re-running the join.
- **`season`** — meteorological season from the discovery date: DJF / MAM / JJA / SON.
- **`season_year`** and **`season_idx`** — the temporal spine. **December is assigned to the winter
  that ends the following calendar year** (Dec 2005 → winter of season-year 2006), so each physical
  winter is one contiguous unit. `season_idx` is a monotonic integer over (season_year, season),
  making "the next season" a `+1` — what the forward-chaining split and persistence baseline need.
  `FIRE_YEAR` is preserved unchanged.

## On `Missing` cause

Missing-cause rows are **kept** in the fire-level file (needed for missingness diagnostics) and
**excluded from the aggregate's shares**, matching `03_missingness.ipynb`'s conclusion: report cause
as shares within attributed fires. The aggregate carries missing acres in a separate column so the
share denominator stays auditable.

## Load the raw table

Every analysis column from `Fires`, dropping only the `Shape` geometry blob.

In [1]:
import sqlite3
import sys

import geopandas as gpd
import numpy as np
import pandas as pd

sys.path.insert(0, "../src")
from cleaning import (
    EcoregionJoiner,
    apply_exclusion,
    build_aggregate,
    derive_temporal_spine,
    exclusion_mask,
    series_support,
)
from config import ProjectConfig

# Paths and the MISSING sentinel come from src/config.py; the transformations from
# src/cleaning.py, unit-tested on synthetic frames covering the December rule, the PR+IA
# overlap, the shared-polygon-edge match, the "(?)" L2 annotation, and the all-missing cell.
cfg = ProjectConfig()
DATA = cfg.data
DB_PATH = cfg.fires_db
CONUS_ECO = cfg.conus_ecoregions
AK_ECO = cfg.ak_ecoregions

FIRE_OUT = cfg.fires_clean               # fire-level, keys attached
AGG_OUT = cfg.region_season_cause        # region x season-year x cause

MISSING = cfg.missing

with sqlite3.connect(DB_PATH) as conn:
    cols = pd.read_sql_query("PRAGMA table_info(Fires);", conn)["name"].tolist()
    load_cols = [c for c in cols if c != "Shape"]
    raw = pd.read_sql_query(f"SELECT {', '.join(load_cols)} FROM Fires", conn)

print(f"Raw: {len(raw):,} fires, {raw.shape[1]} columns.")
print(f"Raw burned area: {raw['FIRE_SIZE'].sum()/1e6:,.1f} million acres")

Raw: 2,303,566 fires, 38 columns.
Raw burned area: 180.0 million acres


## Apply the exclusion

Drop `STATE in ('PR', 'HI')` **or** `NWCG_REPORTING_AGENCY == 'IA'`. The two masks are reported
separately, with their overlap, so the row accounting is transparent.

In [2]:
# No-cause-signal territories plus the IA stream. Alaska is not here: it has no cause-quality
# problem, and its region key comes from the second ecoregion layer joined below.
# `exclusion_mask` returns the combined mask so the two components and their overlap can be
# reported separately.
pr_hi = raw["STATE"].isin(["PR", "HI"])
ia = raw["NWCG_REPORTING_AGENCY"] == "IA"
drop = exclusion_mask(raw)

print(f"PR/HI rows:              {pr_hi.sum():>8,}")
print(f"IA rows:                 {ia.sum():>8,}")
print(f"  of which also PR/HI:   {(pr_hi & ia).sum():>8,}")
print(f"Total rows dropped:      {drop.sum():>8,}  ({drop.mean():.2%} of raw)")
print(f"Acres dropped:           {raw.loc[drop, 'FIRE_SIZE'].sum()/1e6:>8,.2f}M  "
      f"({raw.loc[drop, 'FIRE_SIZE'].sum()/raw['FIRE_SIZE'].sum():.2%} of burned area)")

clean = apply_exclusion(raw)
print(f"\nClean: {len(clean):,} fires  (raw {len(raw):,} - dropped {drop.sum():,}).")

# Alaska retained -- report the acreage at stake.
ak = clean["STATE"] == "AK"
print(f"Alaska retained:         {ak.sum():>8,} fires, "
      f"{clean.loc[ak, 'FIRE_SIZE'].sum()/1e6:.2f}M acres "
      f"({clean.loc[ak, 'FIRE_SIZE'].sum()/clean['FIRE_SIZE'].sum():.1%} of remaining burned area)")

PR/HI rows:                32,172
IA rows:                   21,853
  of which also PR/HI:     21,802
Total rows dropped:        32,223  (1.40% of raw)
Acres dropped:               0.64M  (0.36% of burned area)

Clean: 2,271,343 fires  (raw 2,303,566 - dropped 32,223).
Alaska retained:           15,194 fires, 36.65M acres (20.4% of remaining burned area)


### Confirm the exclusion

No PR/HI/IA rows survive, Alaska does, and the missing-cause rate in the remaining data is the
ordinary case-by-case level rather than the inflated rate the dropped streams carried.

In [3]:
MISSING_RATE = lambda d: (d["NWCG_GENERAL_CAUSE"] == MISSING).mean()

assert not clean["STATE"].isin(["PR", "HI"]).any(), "PR/HI leaked into clean data"
assert not (clean["NWCG_REPORTING_AGENCY"] == "IA").any(), "IA leaked into clean data"
assert (clean["STATE"] == "AK").any(), "Alaska was dropped -- it must be retained"

print(f"Missing-cause rate — raw:   {MISSING_RATE(raw):.1%}")
print(f"Missing-cause rate — clean: {MISSING_RATE(clean):.1%}")
print(f"Missing-cause rate — AK:    {MISSING_RATE(clean[clean['STATE'] == 'AK']):.1%}  "
      "(ordinary level -- AK has no attribution problem)")
print("-> the drop removes the structural non-reporters; what remains is the "
      "ordinary undetermined-cause level.")

Missing-cause rate — raw:   26.0%
Missing-cause rate — clean: 24.9%
Missing-cause rate — AK:    17.5%  (ordinary level -- AK has no attribution problem)
-> the drop removes the structural non-reporters; what remains is the ordinary undetermined-cause level.


## Derive the temporal keys: season, season-year, season index

Meteorological seasons (DJF / MAM / JJA / SON) from the discovery date.

**The December rule.** Meteorological winter spans Dec–Jan–Feb, so December belongs to the winter
that *ends* the following calendar year: Dec 2005 is part of the winter of season-year 2006.
Without this shift each physical winter splits across two calendar years, which makes "predict the
next season" incoherent as a target. 72,106 fires (1.46M acres) are affected.

`season_idx` is a monotonic integer over (season_year, season) — consecutive seasons differ by
exactly 1, so the persistence baseline is a `shift(1)` and the forward-chaining split is a threshold
on this column. `FIRE_YEAR` is left untouched.

**Edge effect:** the shift moves Dec-1992 fires into season-year 1993 and Dec-2020 fires into
season-year 2021, so the first and last winters are partial. Both boundary season-years are reported
below rather than trimmed; the modeling step decides whether to exclude them.

In [4]:
# Derive season / season_year / season_idx from the discovery DATE: DISCOVERY_DOY is complete,
# but meteorological season is defined on the calendar month.
#
# `derive_temporal_spine` owns the December rule (Dec belongs to the winter ending the NEXT
# calendar year), pinned by synthetic tests: 2005-12-15 -> DJF/2006, both record-boundary
# winters, a leap day, and mixed date formats. FIRE_YEAR is preserved; the keys follow the date.
SEASON_ORDER = dict(cfg.season_order)     # kept in scope: the validation cell reconstructs with it
clean = derive_temporal_spine(clean, cfg)

disc = pd.to_datetime(clean["DISCOVERY_DATE"], format="mixed", errors="coerce")
month = disc.dt.month
shifted = (month == 12)
print(f"December fires shifted forward: {shifted.sum():,} "
      f"({clean.loc[shifted, 'FIRE_SIZE'].sum()/1e6:.2f}M acres)")
print(f"season_year span: {clean['season_year'].min()}–{clean['season_year'].max()}")
print(f"season_idx span:  {clean['season_idx'].min()}–{clean['season_idx'].max()} "
      f"({clean['season_idx'].nunique()} distinct seasons present)")

# Contiguity: a gap would break the persistence baseline's shift(1).
gaps = set(range(clean["season_idx"].min(), clean["season_idx"].max() + 1)) - set(clean["season_idx"].unique())
print(f"gaps in season_idx: {sorted(gaps) if gaps else 'none'}")

# Boundary season-years are partial by construction.
for sy in (clean["season_year"].min(), clean["season_year"].max()):
    sub = clean[clean["season_year"] == sy]
    print(f"  season_year {sy}: {len(sub):>8,} fires, seasons present = {sorted(sub['season'].unique())}")

clean[["DISCOVERY_DATE", "FIRE_YEAR", "season", "season_year", "season_idx"]].head()

December fires shifted forward: 70,119 (1.45M acres)
season_year span: 1992–2021
season_idx span:  0–116 (117 distinct seasons present)
gaps in season_idx: none
  season_year 1992:   66,810 fires, seasons present = ['DJF', 'JJA', 'MAM', 'SON']
  season_year 2021:    2,578 fires, seasons present = ['DJF']


,DISCOVERY_DATE,FIRE_YEAR,season,season_year,season_idx
0,2/2/2005,2005,DJF,2005,52
1,5/12/2004,2004,MAM,2004,49
2,5/31/2004,2004,MAM,2004,49
3,6/28/2004,2004,JJA,2004,50
4,6/28/2004,2004,JJA,2004,50


## Derive the region key: two-layer ecoregion join

The region key is the EPA **Level III ecoregion**, and it takes two layers because the standard
shapefile is the *conterminous* US only.

Each join runs **in its own layer's CRS**, with the fire points reprojected to match. Neither
polygon layer is reprojected: CONUS Albers and NAD83 Alaska Albers are tuned to their own
latitudes.

The layers share the same attribute schema on the hierarchy fields, so the results concatenate into
one `region` column at one grain. `na_l2name` is carried for a later Level II roll-up.

Fires are routed by `STATE` — `AK` to the Alaska layer, everything else to CONUS — rather than
joining both layers to all points, which avoids a point matching in both and duplicating a row.

In [5]:
# Two-layer point-in-polygon join, each in its own layer's CRS with the fire points reprojected
# to match. Neither polygon layer is reprojected.
#
# EcoregionJoiner routes AK to the Alaska layer and everything else to CONUS, strips the EPA's
# trailing "(?)" marker from L2 names (left in place it splits one L2 parent into two values and
# a Level II roll-up would double-count), deduplicates any fire matching twice, and raises if an
# L3 region ends up under more than one L2 parent. Those cases are ~0.001% of rows and are pinned
# by synthetic tests on toy polygons in two CRSs.
is_ak = clean["STATE"] == "AK"
clean, crs_info = EcoregionJoiner(CONUS_ECO, AK_ECO).join(clean)
print(f"AK layer CRS:    {crs_info['ak_crs'].name}")
print(f"CONUS layer CRS: {crs_info['conus_crs'].name}")

# Join quality per layer. Acres matter more than rows: the deliverable is acre-weighted.
print()
for name, mask in [("CONUS", ~is_ak), ("Alaska", is_ak)]:
    sub = clean[mask.values]
    ok = sub["region"].notna()
    print(f"{name:6} matched {ok.sum():>9,}/{len(sub):>9,} fires ({ok.mean()*100:5.2f}%)  |  "
          f"acres {sub.loc[ok, 'FIRE_SIZE'].sum()/1e6:7.2f}M/{sub['FIRE_SIZE'].sum()/1e6:7.2f}M "
          f"({sub.loc[ok, 'FIRE_SIZE'].sum()/sub['FIRE_SIZE'].sum()*100:5.2f}%)")

matched = clean["region"].notna()
print(f"\nTotal  matched {matched.sum():>9,}/{len(clean):>9,} fires ({matched.mean()*100:5.2f}%)  |  "
      f"acres {clean.loc[matched, 'FIRE_SIZE'].sum()/1e6:7.2f}M/{clean['FIRE_SIZE'].sum()/1e6:7.2f}M "
      f"({clean.loc[matched, 'FIRE_SIZE'].sum()/clean['FIRE_SIZE'].sum()*100:5.2f}%)")
print(f"distinct ecoregions: {clean['region'].nunique()} "
      f"(L2 parents: {clean['na_l2name'].nunique()})")

# Unmatched points are expected to be coastal/offshore strays.
unmatched_acres = clean.loc[~matched, "FIRE_SIZE"].sum()
print(f"\nunmatched: {(~matched).sum():,} fires, {unmatched_acres/1e6:.3f}M acres "
      f"({unmatched_acres/clean['FIRE_SIZE'].sum()*100:.3f}% of burned area)")
if (~matched).any():
    print("  by state:", clean.loc[~matched, "STATE"].value_counts().head(8).to_dict())

AK layer CRS:    NAD83 / Alaska Albers
CONUS layer CRS: USA_Contiguous_Albers_Equal_Area_Conic_USGS_version

CONUS  matched 2,254,966/2,256,149 fires (99.95%)  |  acres  142.73M/ 142.75M (99.99%)
Alaska matched    14,867/   15,194 fires (97.85%)  |  acres   36.63M/  36.65M (99.93%)

Total  matched 2,269,833/2,271,343 fires (99.93%)  |  acres  179.36M/ 179.41M (99.97%)
distinct ecoregions: 105 (L2 parents: 25)

unmatched: 1,510 fires, 0.045M acres (0.025% of burned area)
  by state: {'AK': 327, 'CA': 179, 'NJ': 165, 'WA': 152, 'FL': 117, 'OR': 101, 'NC': 79, 'MI': 73}


## Build the analysis grain: region × season-year × cause

One row per **region × season_year × season × cause**, carrying burned acres and fire count. This
is the table RQ1 compares and RQ2 predicts.

Two conventions, both following [`03_missingness.ipynb`](03_missingness.ipynb):

- **Shares are computed within attributed fires.** `Missing/undetermined` acres are excluded from
  the share denominator but retained in `missing_acres`, so the denominator is auditable and the
  local missingness rate is available as a per-cell data-quality weight.
- **Cells are dense, not sparse.** A region-season with no fires of a given cause is a real zero,
  not an absent row — the composition has to sum to 1 across the full cause vocabulary. The grid is
  completed explicitly, since a `groupby` alone would leave those combinations missing and distort
  every share.

Only fires with a region are aggregated; the unmatched strays quantified above are dropped.

**Fully unattributed region-seasons produce no row.** The grid is built from region-seasons with at
least one *attributed* fire, so a cell whose fires are 100% missing-cause is absent and its missing
acres are not carried here. The aggregate is the attributed-fire universe, and a cell with no
attributed acres has no cause composition to compare or predict. Their count and acreage are
reported in the validation section. Work from `fires_clean.parquet` for the complete picture.

In [6]:
geo = clean[clean["region"].notna()].copy()
attributed = geo[geo["NWCG_GENERAL_CAUSE"] != MISSING]

KEYS = ["region", "season_year", "season", "season_idx"]

# `build_aggregate` does the groupby, densification against the full cause vocabulary, the
# within-attributed cause_share and the per-cell missing-cause weight, and asserts shares sum
# to 1. Its edge cases -- a densified zero, a cell with no attributed acres, and the
# 100%-missing orphan cell -- are pinned by synthetic tests.
agg = build_aggregate(geo, cfg)

causes = sorted(attributed["NWCG_GENERAL_CAUSE"].unique())
cells = agg[KEYS].drop_duplicates()
miss_total = agg.drop_duplicates(KEYS)["missing_acres"].sum()

print(f"Aggregate: {len(agg):,} rows "
      f"({len(cells):,} region-seasons x {len(causes)} causes)")
print(f"regions: {agg['region'].nunique()} | season-years: {agg['season_year'].nunique()}")
print(f"acres reconciled: {agg['acres'].sum()/1e6:.2f}M attributed + "
      f"{miss_total/1e6:.2f}M missing = "
      f"{(agg['acres'].sum() + miss_total)/1e6:.2f}M "
      f"(geo total {geo['FIRE_SIZE'].sum()/1e6:.2f}M)")
agg.head()

Aggregate: 123,312 rows (10,276 region-seasons x 12 causes)
regions: 105 | season-years: 30
acres reconciled: 146.14M attributed + 33.13M missing = 179.27M (geo total 179.36M)


,region,season_year,season,season_idx,cause,acres,fires,cell_acres,cause_share,missing_acres,missing_fires,missing_acre_frac
0,Acadian Plains and Hills,1992,JJA,2,Arson/incendiarism,15.82,9,36.98,0.427799,9.6,5.0,0.206097
1,Acadian Plains and Hills,1992,JJA,2,Debris and open burning,2.95,4,36.98,0.079773,9.6,5.0,0.206097
2,Acadian Plains and Hills,1992,JJA,2,Equipment and vehicle use,5.80,12,36.98,0.156842,9.6,5.0,0.206097
3,Acadian Plains and Hills,1992,JJA,2,Firearms and explosives use,0.00,0,36.98,0.000000,9.6,5.0,0.206097
4,Acadian Plains and Hills,1992,JJA,2,Fireworks,0.00,0,36.98,0.000000,9.6,5.0,0.206097


### Is Level III a viable grain?

**Hypothesis.** Burned area is tail-driven, so many region-seasons may carry too little history to
predict. If too few series have enough non-empty season-years, the analysis has to coarsen to
Level II.

**Experiment.** For each **region × season** series — the unit RQ2 forecasts through time — count
the season-years carrying attributed fire, and repeat the count at Level II for comparison.

`na_l2name` is already attached, so coarsening would be a `groupby` change, not a re-join.

In [7]:
# Season-years carried by each region x season series, plus the acres each series holds.
n_years = agg["season_year"].nunique()
support = series_support(agg)

print(f"region x season series: {len(support):,}  (max possible years each: {n_years})")
print("\nyears-with-fire distribution across series:")
print(support["years_with_fire"].describe([.1, .25, .5, .75]).round(1).to_string())

for thr in (10, 15, 20, 25):
    ok = support["years_with_fire"] >= thr
    print(f"  series with >= {thr:>2} years of data: {ok.sum():>4}/{len(support)} "
          f"({ok.mean()*100:4.1f}%)  covering {support.loc[ok,'total_acres'].sum()/support['total_acres'].sum()*100:5.1f}% of attributed acres")

# The same question at Level II, the fallback grain.
l2 = geo[geo["NWCG_GENERAL_CAUSE"] != MISSING].groupby(
    ["na_l2name", "season", "season_year"], observed=True)["FIRE_SIZE"].sum().reset_index()
l2_support = (
    l2.groupby(["na_l2name", "season"], observed=True)["season_year"].nunique().reset_index(name="years_with_fire")
)
print(f"\n--- Level II fallback ---")
print(f"region x season series: {len(l2_support):,}")
for thr in (20, 25):
    ok = l2_support["years_with_fire"] >= thr
    print(f"  series with >= {thr} years of data: {ok.sum():>4}/{len(l2_support)} ({ok.mean()*100:4.1f}%)")

print("\nthinnest L3 series (candidates for exclusion or roll-up):")
print(support.nsmallest(8, "years_with_fire")[["region", "season", "years_with_fire", "total_acres"]].to_string(index=False))

region x season series: 402  (max possible years each: 30)

years-with-fire distribution across series:
count    402.0
mean      25.6
std        7.3
min        1.0
10%       15.0
25%       26.2
50%       29.0
75%       29.0
max       30.0
  series with >= 10 years of data:  373/402 (92.8%)  covering  99.9% of attributed acres
  series with >= 15 years of data:  362/402 (90.0%)  covering  99.8% of attributed acres
  series with >= 20 years of data:  346/402 (86.1%)  covering  99.6% of attributed acres
  series with >= 25 years of data:  316/402 (78.6%)  covering  97.9% of attributed acres

--- Level II fallback ---
region x season series: 97
  series with >= 20 years of data:   89/97 (91.8%)
  series with >= 25 years of data:   82/97 (84.5%)

thinnest L3 series (candidates for exclusion or roll-up):
                                      region season  years_with_fire  total_acres
                Ahklun and Kilbuck Mountains    SON                1         20.0
                  Alaska P

## Write the artifacts

`fires_clean.parquet` is fire-level with the keys attached — the base for any diagnostic needing
individual fires. `region_season_cause.parquet` is the analysis grain RQ1 compares and RQ2 predicts.

The fire-level file keeps **all** rows that survived the exclusion, including missing-cause fires
and the small number with no ecoregion match, so downstream work can measure both. The aggregate is
the filtered, modeling-ready view.

In [8]:
FIRE_OUT.parent.mkdir(parents=True, exist_ok=True)

clean.to_parquet(FIRE_OUT, index=False)
print(f"Wrote {len(clean):,} rows x {clean.shape[1]} cols -> {FIRE_OUT}")
print(f"  derived keys: region, na_l2name, season, season_year, season_idx")

agg.to_parquet(AGG_OUT, index=False)
print(f"\nWrote {len(agg):,} rows x {agg.shape[1]} cols -> {AGG_OUT}")
print(f"  grain: region x season_year x season x cause")
print(f"  columns: {list(agg.columns)}")

# Round-trip check: read back what downstream will read.
rt_fires = pd.read_parquet(FIRE_OUT)
rt_agg = pd.read_parquet(AGG_OUT)
assert len(rt_fires) == len(clean) and len(rt_agg) == len(agg), "round-trip row count mismatch"
assert rt_fires["FOD_ID"].is_unique, "FOD_ID not unique in fire-level artifact"
assert not rt_agg.duplicated(KEYS + ["cause"]).any(), "duplicate cells in aggregate"
print("\nround-trip OK.")

Wrote 2,271,343 rows x 43 cols -> /Users/crudman/Documents/GitHub/MSDS696/data/fires_clean.parquet
  derived keys: region, na_l2name, season, season_year, season_idx

Wrote 123,312 rows x 12 cols -> /Users/crudman/Documents/GitHub/MSDS696/data/region_season_cause.parquet
  grain: region x season_year x season x cause
  columns: ['region', 'season_year', 'season', 'season_idx', 'cause', 'acres', 'fires', 'cell_acres', 'cause_share', 'missing_acres', 'missing_fires', 'missing_acre_frac']

round-trip OK.


## Validation

A final pass over the two files **as read from disk**, independent of the in-memory objects above.
Everything the rest of the project assumes about these artifacts is asserted here.

- **Exclusion rule** — no PR/HI/IA rows survive; Alaska does.
- **Temporal spine** — season labels are the four expected values, `season_idx` is contiguous and
  reconstructible from `(season_year, season)`, and the December rule moved December fires forward.
  Checked against the **parsed `DISCOVERY_DATE`**, which is what the keys derive from — not
  `FIRE_YEAR`. The two disagree for a handful of source records (a December 31 discovery filed under
  the next year's `FIRE_YEAR`); that is a source quirk, reported as a count rather than asserted on.
- **Region key** — every region carries exactly one `na_l2name` parent, so a Level II roll-up is
  well defined, and the join lost a negligible share of acres.
- **Aggregate grain** — one row per `region × season_year × season × cause`, dense over the full
  cause vocabulary, no `Missing` in the cause column, shares sum to 1 wherever attributed acres exist.
- **Cross-artifact reconciliation** — the aggregate's attributed acres, missing acres and fire counts
  add back up to the fire-level file for the same population, checked **per cell** rather than only
  in total, since a compensating error in the densify step would pass a grand-total check.

**Orphan cells.** A region-season whose fires are entirely missing-cause has no row in the
aggregate, so its missing acres appear nowhere there. The reconciliation prints their count and
acreage and asserts they stay a negligible share of burned area.

In [9]:
# Validate the files as read from disk, not the in-memory frames.
F = pd.read_parquet(FIRE_OUT)
A = pd.read_parquet(AGG_OUT)

results = []

def check(name, condition, detail=""):
    """Record a named assertion; collect all failures instead of stopping at the first."""
    results.append((bool(condition), name, detail))

# --- schema -------------------------------------------------------------------
FIRE_REQUIRED = ["FOD_ID", "FIRE_YEAR", "STATE", "FIRE_SIZE", "NWCG_GENERAL_CAUSE",
                 "region", "na_l2name", "season", "season_year", "season_idx"]
AGG_REQUIRED = ["region", "season_year", "season", "season_idx", "cause", "acres", "fires",
                "cell_acres", "cause_share", "missing_acres", "missing_fires",
                "missing_acre_frac"]
check("fire-level: required columns present",
      set(FIRE_REQUIRED) <= set(F.columns),
      f"missing {sorted(set(FIRE_REQUIRED) - set(F.columns))}")
check("aggregate: required columns present",
      set(AGG_REQUIRED) <= set(A.columns),
      f"missing {sorted(set(AGG_REQUIRED) - set(A.columns))}")
check("fire-level: FOD_ID unique and non-null",
      F["FOD_ID"].is_unique and F["FOD_ID"].notna().all())
check("fire-level: no null keys except region on unmatched strays",
      F[["season", "season_year", "season_idx", "FIRE_SIZE"]].notna().all().all())

# --- exclusion rule -----------------------------------------------------------
check("exclusion: no PR/HI rows", not F["STATE"].isin(["PR", "HI"]).any())
check("exclusion: no IA reporting stream", not (F["NWCG_REPORTING_AGENCY"] == "IA").any())
check("exclusion: Alaska retained", (F["STATE"] == "AK").sum() > 0,
      f"{(F['STATE'] == 'AK').sum():,} AK fires")
check("exclusion: row count matches raw minus documented drop",
      len(F) == len(raw) - int(drop.sum()), f"{len(F):,}")

# --- temporal spine -----------------------------------------------------------
check("season: exactly the four meteorological labels",
      set(F["season"].unique()) == {"DJF", "MAM", "JJA", "SON"})
idx_recon = ((F["season_year"] - F["season_year"].min()) * 4
             + F["season"].map(SEASON_ORDER))
check("season_idx: reconstructible from (season_year, season)",
      (idx_recon == F["season_idx"]).all())
present = np.sort(F["season_idx"].unique())
check("season_idx: contiguous (no gap breaks the +1 next-season step)",
      np.array_equal(present, np.arange(present.min(), present.max() + 1)),
      f"span {present.min()}-{present.max()}, {len(present)} present")
check("season_idx: monotone in (season_year, season)",
      F.groupby("season_idx")[["season_year"]].nunique().eq(1).all().all())
# season/season_year derive from DISCOVERY_DATE, so they are checked against the parsed date,
# not FIRE_YEAR. The two disagree for a handful of source records (e.g. a 12/31/2006 discovery
# filed under FIRE_YEAR 2007) -- a source quirk, reported rather than asserted on.
d_month = pd.to_datetime(F["DISCOVERY_DATE"], format="mixed", errors="coerce")
dec = d_month.dt.month == 12
check("December rule: all December fires land in DJF of the following calendar year",
      (F.loc[dec, "season"] == "DJF").all()
      and (F.loc[dec, "season_year"] == d_month.dt.year[dec] + 1).all(),
      f"{dec.sum():,} December fires")
check("non-December fires: season_year equals the discovery calendar year",
      (F.loc[~dec, "season_year"] == d_month.dt.year[~dec]).all())
fy_mismatch = (F["season_year"] != d_month.dt.year + dec.astype(int))
check("season_year is a pure function of DISCOVERY_DATE", not fy_mismatch.any())
n_fy_odd = int((F["FIRE_YEAR"] != d_month.dt.year).sum())
print(f"note: {n_fy_odd} source rows where FIRE_YEAR != discovery calendar year "
      "(source quirk; season keys follow DISCOVERY_DATE)\n")

# --- region key ---------------------------------------------------------------
geo_f = F[F["region"].notna()]
check("region: every region maps to exactly one Level II parent",
      geo_f.groupby("region")["na_l2name"].nunique().eq(1).all())
check("region: na_l2name present wherever region is",
      geo_f["na_l2name"].notna().all())
matched_acres = geo_f["FIRE_SIZE"].sum() / F["FIRE_SIZE"].sum()
check("region: spatial join retains >=99.9% of burned acres",
      matched_acres >= 0.999, f"{matched_acres:.4%}")
check("region: Alaska fires carry a region (two-layer join worked)",
      F.loc[F["STATE"] == "AK", "region"].notna().mean() > 0.97,
      f"{F.loc[F['STATE'] == 'AK', 'region'].notna().mean():.2%} of AK fires")

# --- aggregate grain ----------------------------------------------------------
check("aggregate: one row per region x season_year x season x cause",
      not A.duplicated(KEYS + ["cause"]).any())
check("aggregate: Missing cause excluded from the cause column",
      MISSING not in set(A["cause"].unique()))
n_causes = A["cause"].nunique()
check("aggregate: dense -- every cell carries the full cause vocabulary",
      A.groupby(KEYS, observed=True)["cause"].size().eq(n_causes).all(),
      f"{n_causes} causes x {len(A) // n_causes:,} cells")
check("aggregate: no negative acres or counts",
      (A[["acres", "fires", "cell_acres", "missing_acres", "missing_fires"]] >= 0).all().all())
nonempty = A[A["cell_acres"] > 0]
check("aggregate: cause_share sums to 1 in every non-empty cell",
      np.allclose(nonempty.groupby(KEYS, observed=True)["cause_share"].sum(), 1.0))
check("aggregate: cause_share within [0, 1]",
      nonempty["cause_share"].between(0, 1).all())
check("aggregate: cause_share is NaN exactly where a cell has no attributed acres",
      A["cause_share"].isna().equals(A["cell_acres"] <= 0))
check("aggregate: cell_acres equals the cell's summed cause acres",
      np.allclose(A.groupby(KEYS, observed=True)["acres"].transform("sum"), A["cell_acres"]))
check("aggregate: missing_acre_frac within [0, 1] where defined",
      A["missing_acre_frac"].dropna().between(0, 1).all())
check("aggregate: missing_* constant within a cell",
      A.groupby(KEYS, observed=True)[["missing_acres", "missing_fires"]].nunique().eq(1).all().all())

# --- cross-artifact reconciliation --------------------------------------------
# Rebuild the aggregate's totals from the fire-level file and compare -- ties the two
# artifacts together rather than each to itself.
src = F[F["region"].notna()]
src_attr = src[src["NWCG_GENERAL_CAUSE"] != MISSING]
src_miss = src[src["NWCG_GENERAL_CAUSE"] == MISSING]

check("reconcile: attributed acres agree",
      np.isclose(A["acres"].sum(), src_attr["FIRE_SIZE"].sum()),
      f"agg {A['acres'].sum()/1e6:.3f}M vs fires {src_attr['FIRE_SIZE'].sum()/1e6:.3f}M")
check("reconcile: attributed fire counts agree",
      A["fires"].sum() == len(src_attr),
      f"agg {A['fires'].sum():,} vs fires {len(src_attr):,}")

# Known gap: the grid is built from region-seasons with at least one attributed fire, so a
# 100%-missing region-season has no row and its missing acres are carried nowhere. The excluded
# volume is quantified below, and the reconciliation runs over the cells the aggregate covers.
cell_key = A[KEYS + ["missing_acres", "missing_fires"]].drop_duplicates(KEYS)
miss_by_cell = (src_miss.groupby(KEYS, observed=True)["FIRE_SIZE"].sum()
                .rename("src_missing").reset_index())
miss_cmp = cell_key.merge(miss_by_cell, on=KEYS, how="outer", indicator=True)
orphan = miss_cmp[miss_cmp["_merge"] == "right_only"]
print(f"note: {len(orphan):,} region-seasons are 100% missing-cause and are absent from the "
      f"aggregate by design\n      ({orphan['src_missing'].sum():,.0f} acres, "
      f"{orphan['src_missing'].sum()/src['FIRE_SIZE'].sum()*100:.3f}% of geo-matched acres)\n")

shared = miss_cmp[miss_cmp["_merge"] == "both"]
check("reconcile: missing acres agree on every cell the aggregate covers",
      np.allclose(shared["missing_acres"], shared["src_missing"]),
      f"{len(shared):,} cells compared")
check("reconcile: no aggregate cell invents missing acres the fire file lacks",
      np.isclose(miss_cmp.loc[miss_cmp["_merge"] == "left_only", "missing_acres"].fillna(0).sum(), 0.0))
check("reconcile: acres close over the aggregate's cells (attributed + missing + orphan)",
      np.isclose(A["acres"].sum() + cell_key["missing_acres"].sum()
                 + orphan["src_missing"].sum(), src["FIRE_SIZE"].sum()),
      f"{(A['acres'].sum() + cell_key['missing_acres'].sum() + orphan['src_missing'].sum())/1e6:.3f}M "
      f"vs {src['FIRE_SIZE'].sum()/1e6:.3f}M")
check("reconcile: the orphan cells are a negligible share of burned area",
      orphan["src_missing"].sum() / src["FIRE_SIZE"].sum() < 0.001,
      f"{orphan['src_missing'].sum()/src['FIRE_SIZE'].sum():.4%}")

# Per-cell, not just in total: a compensating densify error would pass a grand-total check.
by_cell = (src_attr.groupby(KEYS, observed=True)["FIRE_SIZE"].sum()
           .rename("src_acres").reset_index())
merged = cell_key[KEYS].merge(by_cell, on=KEYS, how="outer", indicator=True)
check("reconcile: aggregate covers exactly the non-empty region-seasons in the fire file",
      (merged["_merge"] == "both").all(),
      f"{(merged['_merge'] != 'both').sum()} mismatched cells")
per_cell = (A.groupby(KEYS, observed=True)["acres"].sum().rename("agg_acres").reset_index()
            .merge(by_cell, on=KEYS, how="inner"))
check("reconcile: per-cell attributed acres agree for every region-season",
      np.allclose(per_cell["agg_acres"], per_cell["src_acres"]),
      f"{len(per_cell):,} cells compared")

check("reconcile: aggregate regions are a subset of fire-level regions",
      set(A["region"]) <= set(geo_f["region"]))
check("reconcile: aggregate cause vocabulary matches the attributed fire causes",
      set(A["cause"]) == set(src_attr["NWCG_GENERAL_CAUSE"].unique()))

# --- report -------------------------------------------------------------------
failed = [r for r in results if not r[0]]
width = max(len(n) for _, n, _ in results)
for ok, name, detail in results:
    print(f"[{'PASS' if ok else 'FAIL'}] {name:<{width}}  {detail}")
print(f"\n{len(results) - len(failed)}/{len(results)} checks passed.")
assert not failed, "VALIDATION FAILED:\n" + "\n".join(f"  - {n} {d}" for _, n, d in failed)
print("Both artifacts validated -- safe for downstream use.")

note: 38 source rows where FIRE_YEAR != discovery calendar year (source quirk; season keys follow DISCOVERY_DATE)

note: 170 region-seasons are 100% missing-cause and are absent from the aggregate by design
      (92,669 acres, 0.052% of geo-matched acres)

[PASS] fire-level: required columns present                                               missing []
[PASS] aggregate: required columns present                                                missing []
[PASS] fire-level: FOD_ID unique and non-null                                             
[PASS] fire-level: no null keys except region on unmatched strays                         
[PASS] exclusion: no PR/HI rows                                                           
[PASS] exclusion: no IA reporting stream                                                  
[PASS] exclusion: Alaska retained                                                         15,194 AK fires
[PASS] exclusion: row count matches raw minus documented drop         